# Normalizing Flow Model Validation

Interactive notebook for visualizing and validating normalizing flow models trained on FAIR Universe data.

In [1]:
import pkgutil
print(any(m.name == "fair_universe_demo" for m in pkgutil.iter_modules()))

False


In [2]:
import fair_universe_demo

ModuleNotFoundError: No module named 'fair_universe_demo'

In [ ]:
import matplotlib.pyplot as plt
from typing import List
import numpy as np
import torch
import os
import sys

sys.path.append("../..")

from fair_universe_demo.models.classifier_datamodule import ClassifierDatamodule
from tasks.histogram import HistogramTask
from utils.selection import createJetData

ModuleNotFoundError: No module named 'fair_universe_demo'

## Configuration

In [ ]:
# Global Configuration Variables
snapshot_path = "path/to/snapshot.json"  # Update with actual path
root_dir = "path/to/fair_universe_data"  # Update with actual path
plot_save_dir = "./validation_plots"

# Global data storage (will be populated in setup)
signal_data = None
bg_data = None
signal_logprobs = None
bg_logprobs = None
nf_model = None
model_name = None
device = "cuda" if torch.cuda.is_available() else "cpu"

## Setup and Data Loading

In [ ]:
def setup():
    """Load NF models from snapshot and prepare data for the first model."""
    global signal_data, bg_data, signal_logprobs, bg_logprobs, nf_model, model_name, device
    
    # Parse snapshot to get NF models
    nf_ckpts, _ = HistogramTask.parse_snapshot(snapshot_path)
    nf_models = ClassifierDatamodule.load_nf_models(nf_ckpts)
    
    # Get the first model only
    model_name, nf_model = list(nf_models.items())[0]
    print(f"Loading model: {model_name}")
    
    nf_model = nf_model.to(device).eval()
    
    os.makedirs(plot_save_dir, exist_ok=True)
    
    # Extract jet count from model name (e.g., "nf_signal_1jet&c_0p5" -> 1 or 2)
    name_parts = model_name.split('&')[0]  # "nf_signal_1jet" or "nf_background_2jet"
    if '1jet' in name_parts:
        num_jets = 1
    elif '2jet' in name_parts:
        num_jets = 2
    else:
        raise ValueError(f"Could not extract jet count from model name: {model_name}")
    
    # Load validation data
    print(f"Loading validation data for {num_jets} jets...")
    data, detlabel, _, _ = createJetData(
        jet_num=num_jets,
        useTestData=False,
        seed=78,
        root_dir=root_dir,
    )
    
    # Split by signal (1) and background (0)
    signal_mask = detlabel == 1
    bg_mask = detlabel == 0
    
    signal_data = data[signal_mask]
    bg_data = data[bg_mask]
    
    # Evaluate model on both signal and background
    print("Evaluating model on signal and background data...")
    with torch.no_grad():
        signal_logprobs = nf_model(signal_data.to(device)).cpu().numpy()
        bg_logprobs = nf_model(bg_data.to(device)).cpu().numpy()
    
    print(f"Loaded {len(signal_logprobs)} signal and {len(bg_logprobs)} background samples")
    print(f"Signal log-probs: mean={np.mean(signal_logprobs):.3f}, std={np.std(signal_logprobs):.3f}")
    print(f"Background log-probs: mean={np.mean(bg_logprobs):.3f}, std={np.std(bg_logprobs):.3f}")

# Run setup
setup()

## Plot 1: Log-Probability Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram comparison
axes[0].hist(signal_logprobs, bins=50, alpha=0.6, label='Signal', density=True)
axes[0].hist(bg_logprobs, bins=50, alpha=0.6, label='Background', density=True)
axes[0].set_xlabel('Log-Probability')
axes[0].set_ylabel('Density')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Quantile comparison
q_sig = np.quantile(signal_logprobs, np.linspace(0, 1, 100))
q_bg = np.quantile(bg_logprobs, np.linspace(0, 1, 100))
axes[1].plot(q_sig, label='Signal', linewidth=2)
axes[1].plot(q_bg, label='Background', linewidth=2)
axes[1].set_xlabel('Quantile Index')
axes[1].set_ylabel('Log-Probability Value')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Save figure
fig_path = os.path.join(plot_save_dir, f"{model_name}_log_prob_distribution.png")
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f"Saved: {fig_path}")

## Plot 2: Log-Probability Statistics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Box plot
data_to_plot = [signal_logprobs, bg_logprobs]
bp = axes[0].boxplot(data_to_plot, labels=['Signal', 'Background'], patch_artist=True)
for patch, color in zip(bp['boxes'], ['lightblue', 'lightcoral']):
    patch.set_facecolor(color)
axes[0].set_ylabel('Log-Probability')
axes[0].grid(True, alpha=0.3, axis='y')

# Statistics table
stats = {
    'Signal': {
        'Mean': np.mean(signal_logprobs),
        'Std': np.std(signal_logprobs),
        'Median': np.median(signal_logprobs),
        'Min': np.min(signal_logprobs),
        'Max': np.max(signal_logprobs),
    },
    'Background': {
        'Mean': np.mean(bg_logprobs),
        'Std': np.std(bg_logprobs),
        'Median': np.median(bg_logprobs),
        'Min': np.min(bg_logprobs),
        'Max': np.max(bg_logprobs),
    },
}

axes[1].axis('off')
table_data = [['Metric', 'Signal', 'Background']]
for metric in ['Mean', 'Std', 'Median', 'Min', 'Max']:
    table_data.append([
        metric,
        f"{stats['Signal'][metric]:.4f}",
        f"{stats['Background'][metric]:.4f}",
    ])

table = axes[1].table(
    table_data,
    cellLoc='center',
    loc='center',
    colWidths=[0.3, 0.35, 0.35],
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2)

plt.tight_layout()
plt.show()

# Save figure
fig_path = os.path.join(plot_save_dir, f"{model_name}_log_prob_statistics.png")
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f"Saved: {fig_path}")

## Plot 3: Log-Probability vs Feature

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

sig_data_np = signal_data.cpu().numpy()
bg_data_np = bg_data.cpu().numpy()

for i in range(min(4, 4)):
    axes[i].scatter(sig_data_np[:, i], signal_logprobs, alpha=0.3, s=5, label='Signal')
    axes[i].scatter(bg_data_np[:, i], bg_logprobs, alpha=0.3, s=5, label='Background')
    axes[i].set_xlabel(f'Feature {i}')
    axes[i].set_ylabel('Log-Probability')
    axes[i].set_title(f'Log-Prob vs Feature {i}')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Save figure
fig_path = os.path.join(plot_save_dir, f"{model_name}_log_prob_vs_feature.png")
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f"Saved: {fig_path}")

## Plot 4: Training Curves (Optional)

In [ ]:
# Example training curves (replace with actual training history if available)
def plot_training_curves(train_losses: List[float], val_losses: List[float], title: str = "Training Curves"):
    fig, ax = plt.subplots(figsize=(10, 6))
    
    epochs = np.arange(len(train_losses))
    ax.plot(epochs, train_losses, label='Training Loss', linewidth=2)
    ax.plot(epochs, val_losses, label='Validation Loss', linewidth=2)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    return fig

# Example usage (uncomment if you have training data):
# train_losses = [...]  # Your training losses
# val_losses = [...]    # Your validation losses
# fig = plot_training_curves(train_losses, val_losses)
# plt.show()
# fig_path = os.path.join(plot_save_dir, f"{model_name}_training_curves.png")
# fig.savefig(fig_path, dpi=150, bbox_inches='tight')

## Plot 5: Calibration Curve (ROC)

In [ ]:
num_bins = 20
thresholds = np.linspace(
    min(signal_logprobs.min(), bg_logprobs.min()),
    max(signal_logprobs.max(), bg_logprobs.max()),
    num_bins,
)

sig_eff = []  # Fraction of signal above threshold (true positive rate)
bg_acc = []   # Fraction of background above threshold (false positive rate)

for thresh in thresholds:
    sig_eff.append(np.mean(signal_logprobs > thresh))
    bg_acc.append(np.mean(bg_logprobs > thresh))

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(
    bg_acc,
    sig_eff,
    linewidth=2,
    marker='o',
    markersize=5,
    label="Contrastive Normalizing Flow",
)
ax.set_xlabel('Background Acceptance (false positive rate)')
ax.set_ylabel('Signal Efficiency (true positive rate)')
ax.grid(True, alpha=0.3)
ax.set_xlim(*[0, 1])
ax.set_ylim(*[0, 1])

# Diagonal reference line
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, linewidth=1, label='Random classifier')
ax.legend()

plt.tight_layout()
plt.show()

# Save figure
fig_path = os.path.join(plot_save_dir, f"{model_name}_calibration_curve.png")
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f"Saved: {fig_path}")

## Plot 6: Feature Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

sig_data_np = signal_data.cpu().numpy()
bg_data_np = bg_data.cpu().numpy()

for i in range(min(4, 4)):
    axes[i].hist(sig_data_np[:, i], bins=50, alpha=0.6, label='Signal', density=True)
    axes[i].hist(bg_data_np[:, i], bins=50, alpha=0.6, label='Background', density=True)
    axes[i].set_xlabel(f'Feature {i}')
    axes[i].set_ylabel('Density')
    axes[i].set_title(f'Feature {i} Distribution')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Save figure
fig_path = os.path.join(plot_save_dir, f"{model_name}_feature_distribution.png")
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f"Saved: {fig_path}")
print(f"\nAll plots saved to: {plot_save_dir}")